# 03. Other Models: SVM, Random Forest, and 2GDNN

This notebook evaluates additional classification models on the clinical dataset. It covers Support Vector Machines (SVM), Tree-based ensemble methods (Random Forest with Grid Search), and Deep Learning (a 2-Layer Deep Neural Network).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../')

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from src.data_processing import load_and_clean_data, get_train_test_data
from src.evaluation import print_model_performance, plot_confusion_matrix

plt.style.use('seaborn-v0_8')
%matplotlib inline


## Data Loading and Preprocessing

In [ ]:
# Using mock data to demonstrate the pipeline. 
# In production, replace with: df = load_and_clean_data('../data/raw/data_pima.csv')
np.random.seed(42)
df = pd.DataFrame({
    'Glucose': np.random.normal(120, 30, 500),
    'BloodPressure': np.random.normal(70, 10, 500),
    'Insulin': np.random.exponential(50, 500),
    'BMI': np.random.normal(30, 5, 500),
    'Age': np.random.randint(21, 80, 500),
    'Outcome': np.random.randint(0, 3, 500)  # Multi-class for demonstration based on Final.ipynb
})

X_train, X_test, y_train, y_test = get_train_test_data(df, target_col='Outcome')
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

## 1. Support Vector Machine (SVM)
SVM performs well on high-dimensional spaces and is effective when there is a clear margin of separation. We use the RBF kernel here.

In [ ]:
svm_model = SVC(kernel='rbf', C=10.0, gamma='scale', random_state=42)
svm_model.fit(X_train, y_train)
svm_pred = svm_model.predict(X_test)

print_model_performance(y_test, svm_pred, model_name="Support Vector Machine (SVM)")
plot_confusion_matrix(y_test, svm_pred, title="SVM Confusion Matrix")

## 2. Random Forest with GridSearchCV
Random Forests are robust to overfitting. We use `GridSearchCV` to find the optimal hyperparameters for the tree depth and number of estimators.

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7, None],
    'max_features': ['sqrt', 'log2']
}

rf_base = RandomForestClassifier(random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(estimator=rf_base, param_grid=param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best Hyperparameters: {grid_search.best_params_}")
rf_best = grid_search.best_estimator_
rf_pred = rf_best.predict(X_test)

print_model_performance(y_test, rf_pred, model_name="Random Forest (Optimized)")

## 3. Deep Neural Network (2GDNN)
We construct a 2-Layer Deep Neural Network using Keras to capture complex non-linear patterns in the clinical data.

In [ ]:
# Convert labels to categorical for categorical_crossentropy
from keras.utils import to_categorical
num_classes = len(np.unique(y_train))
y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

def build_2gdnn(input_shape, num_classes):
    model = keras.Sequential([
        layers.Dense(16, activation='relu', input_shape=(input_shape,)),
        layers.Dropout(0.2),
        layers.Dense(8, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', 
                  loss='categorical_crossentropy', 
                  metrics=['accuracy'])
    return model

dnn_model = build_2gdnn(X_train.shape[1], num_classes)
dnn_model.summary()

In [ ]:
# Train the model
history = dnn_model.fit(
    X_train, y_train_cat,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    verbose=0  # Set to 0 to keep the notebook clean
)

# Evaluate
dnn_loss, dnn_acc = dnn_model.evaluate(X_test, y_test_cat, verbose=0)
print(f"\n2GDNN Test Accuracy: {dnn_acc:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['loss'], label='Train Loss')
ax1.plot(history.history['val_loss'], label='Validation Loss')
ax1.set_title('DNN Loss over Epochs')
ax1.legend()

ax2.plot(history.history['accuracy'], label='Train Accuracy')
ax2.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax2.set_title('DNN Accuracy over Epochs')
ax2.legend()

plt.show()